# Oliver Assessment Laboratory

Use the real authenticated **Admin API → Assessment Agent → deterministic policy → Coach Agent** path to evaluate an idea and preview the complete Oliver reply. This notebook does not import or reproduce Oliver's scoring rules, and it does not persist an initiative, evidence version, assessment, lifecycle transition, delivery job, or email.

Before running it, keep the local services available at `http://localhost:8000` and `http://localhost:8001`. Model processing can take several minutes.

In [ ]:
import getpass
import html
import os
from typing import Any

import httpx
from IPython.display import HTML, Markdown, display

ADMIN_API_URL = os.getenv("OLIVER_ADMIN_API_URL", "http://localhost:8000").rstrip("/")
username = input("Local admin username: ").strip()
password = getpass.getpass("Local admin password: ")

client = httpx.Client(base_url=ADMIN_API_URL, timeout=700.0, follow_redirects=True)
health = client.get("/health")
health.raise_for_status()
login = client.post("/api/v1/auth/login", json={"username": username, "password": password})
password = ""
login.raise_for_status()
session = login.json()
required_role = "Oliver.Assessment.Test"
if required_role not in session["roles"] and "Oliver.Platform.Admin" not in session["roles"]:
    raise PermissionError(f"Signed-in identity does not have {required_role}")

print(f"Connected to {ADMIN_API_URL} as {session['username']}")
print(f"Roles: {', '.join(session['roles'])}")

## Describe your idea

Replace the sample below with your real idea. Include the problem, affected users or process, expected value, available evidence/data, technical approach, sponsor or owner, delivery capacity, risks, dependencies, and any measured results. Do not include secrets or restricted personal data.

In [ ]:
idea = {
    "subject": "Turbine Inspection AI Pilot Update",
    "current_stage": "DI1",
    "evidence": """
Hi Oliver,

A quick update on our turbine inspection AI pilot.

In our previous testing, the model achieved around 92% defect detection accuracy and the initial feedback from the engineering team was positive.

We have now completed a second validation using a newer set of inspection images. Performance dropped to around 74%, with most of the missed cases coming from one defect category that was not well represented in the original test data.

The engineering team believes we can improve this by adding more representative training examples, but we have not tested that yet.

We are still interested in continuing the pilot. Based on the updated evidence, would you recommend that we move forward with the next stage or address the performance issue first?

Best regards,
Project Team
""".strip(),
}

idea


In [ ]:
def run_assessment(payload: dict[str, str]) -> tuple[dict[str, Any], int, int]:
    before = client.get("/api/v1/initiatives")
    before.raise_for_status()
    before_count = len(before.json())

    response = client.post("/api/v1/commands/assessment-test", json=payload)
    if response.is_error:
        try:
            detail = response.json().get("detail", response.text)
        except ValueError:
            detail = response.text
        raise RuntimeError(f"Assessment failed ({response.status_code}): {detail}")

    after = client.get("/api/v1/initiatives")
    after.raise_for_status()
    return response.json(), before_count, len(after.json())

assessment, initiatives_before, initiatives_after = run_assessment(idea)
print("Assessment completed.")

In [ ]:
def safe(value: object) -> str:
    return html.escape(str(value), quote=True).replace("|", "&#124;").replace("\n", " ")

score = assessment["composite_score"] if assessment["composite_score"] is not None else "—"
summary = [
    "## Policy result",
    "",
    f"- **Canonical score:** {safe(score)}",
    f"- **Rating:** {safe(assessment['rating'])}",
    f"- **Gate outcome:** {safe(assessment['gate_outcome'])}",
    f"- **Current stage:** {safe(assessment['current_stage'])}",
    f"- **Transition target:** {safe(assessment['transition_target'])}",
    f"- **Recommended next stage:** {safe(assessment['recommended_next_stage'])}",
    f"- **Human review required:** {safe(assessment['requires_human_review'])}",
    f"- **Evaluator:** {safe(assessment['model_version'])}",
    f"- **Score policy:** {safe(assessment['weight_set_version'])}",
    f"- **Transition policy:** {safe(assessment['transition_policy_version'])}",
    "",
    safe(assessment["transition_rationale"]),
    "",
    "## Evidence dimensions",
    "",
    "| Dimension | Score | Confidence | Weight |",
    "|---|---:|---:|---:|",
]
for dimension in assessment["dimensions"]:
    summary.append(
        f"| {safe(dimension['dimension_label'])} | {safe(dimension['value'] if dimension['value'] is not None else dimension['state'])} | "
        f"{dimension['confidence']:.0%} | {dimension['weight']}% |"
    )
display(Markdown("\n".join(summary)))

for dimension in assessment["dimensions"]:
    dimension_result = (
        f"{dimension['value']}/100"
        if dimension["value"] is not None
        else dimension["state"].replace("_", " ").title()
    )
    evidence = "; ".join(safe(item) for item in dimension["evidence"]) or "No supporting evidence identified."
    gaps = "; ".join(safe(item) for item in dimension["gaps"]) or "No material gap identified."
    display(
        Markdown(
            f"### {safe(dimension['dimension_label'])} — {safe(dimension_result)}\n\n"
            f"{safe(dimension['summary'])}\n\n"
            f"**Evidence:** {evidence}\n\n"
            f"**Gaps:** {gaps}"
        )
    )

## Complete Oliver reply

This is the full branded email Oliver would return to the original submitter. The sandbox renders it but does not store, queue, or send it.

In [ ]:
if assessment["response_action"] != "SEND_EMAIL" or not assessment.get("email_html"):
    display(Markdown("Oliver decided that no reply should be sent for this test input."))
else:
    display(Markdown(f"**Email subject:** {safe(assessment['email_subject'])}"))
    email_document = assessment["email_html"] or ""
    preview_csp = (
        "<meta http-equiv=\"Content-Security-Policy\" "
        "content=\"default-src 'none'; img-src data:; style-src 'unsafe-inline'; "
        "base-uri 'none'; form-action 'none'\">"
    )
    email_document = email_document.replace("<head>", f"<head>{preview_csp}", 1)
    preview_document = html.escape(email_document, quote=True)
    display(HTML(f'<iframe title="Oliver email preview" sandbox="" srcdoc="{preview_document}" style="width:100%; height:1000px; border:1px solid #b8c2c8;"></iframe>'))


In [ ]:
assert initiatives_after == initiatives_before, (
    f"The sandbox changed the initiative count from {initiatives_before} to {initiatives_after}."
)
print(f"Initiative-count check passed: count remained {initiatives_after}.")

## Finish

Edit the `idea` cell and rerun the assessment and display cells for another non-persistent test. Run the cleanup cell below when finished.

In [ ]:
logout = client.post("/api/v1/auth/logout")
logout.raise_for_status()
client.close()
print("Local admin session closed.")